In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilasi sirkuit dari jarak jauh dengan Qiskit Transpiler Service

> **Danger:** Per 18 Juli 2025, layanan ini sedang dimigrasikan untuk mendukung IBM Quantum&reg; Platform yang baru dan saat ini tidak tersedia. Untuk AI passes, kamu bisa menggunakan [mode lokal](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     Layanan ini adalah rilis beta yang bisa berubah sewaktu-waktu.
>     Kalau kamu punya masukan atau ingin menghubungi tim pengembang, silakan gunakan channel [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) ini.

Qiskit Transpiler Service menyediakan kemampuan transpilasi di cloud. Selain kemampuan Transpiler Qiskit lokal, tugas transpilasi kamu bisa memanfaatkan sumber daya cloud IBM Quantum dan AI transpiler passes yang didukung kecerdasan buatan.

Qiskit Transpiler Service menawarkan library Python untuk mengintegrasikan layanan ini dan kemampuannya secara mulus ke dalam pola dan alur kerja Qiskit kamu saat ini. Layanan ini hanya tersedia untuk pengguna IBM Quantum Premium Plan, Flex Plan, dan On-Prem (melalui IBM Quantum Platform API) Plan.

<span id="install-transpiler-service"></span>
## Instal paket qiskit-ibm-transpiler
Untuk menggunakan Qiskit Transpiler Service, instal paket `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Paket ini secara otomatis melakukan autentikasi menggunakan [kredensial IBM Quantum Platform](/guides/cloud-setup) kamu, sesuai dengan cara [Qiskit Runtime mengelolanya](/guides/initialize-account):
- Variabel lingkungan: `QISKIT_IBM_TOKEN`
- File konfigurasi `~/.qiskit/qiskit-ibm.json` (di bawah bagian `default-ibm-quantum`).

*Catatan*: Paket ini memerlukan Qiskit SDK v1.X.

## Opsi transpilasi qiskit-ibm-transpiler
- `backend_name` (opsional, str) - Nama Backend sebagaimana yang diharapkan oleh QiskitRuntimeService (misalnya, `ibm_torino`). Jika ini diatur, metode transpile menggunakan layout dari Backend yang ditentukan untuk operasi transpilasi. Jika ada opsi lain yang diatur yang memengaruhi pengaturan ini, seperti `coupling_map`, pengaturan `backend_name` akan di-override.
- `coupling_map` (opsional, List[List[int]]) - Daftar coupling map yang valid (misalnya, [[0,1],[1,2]]). Jika ini diatur, metode transpile menggunakan coupling map ini untuk operasi transpilasi. Jika didefinisikan, nilainya akan meng-override nilai yang ditentukan untuk `target`.
- `optimization_level` (int) - Level optimasi potensial yang diterapkan selama proses transpilasi. Nilai yang valid adalah [1,2,3], di mana 1 adalah optimasi paling sedikit (dan tercepat), dan 3 adalah optimasi paling banyak (dan paling memakan waktu).
- `ai` ("true", "false", "auto") - Apakah akan menggunakan kemampuan berbasis AI selama transpilasi. Kemampuan berbasis AI yang tersedia bisa berupa `AIRouting` transpiling passes atau metode sintesis berbasis AI lainnya. Jika nilainya `"true"`, layanan menerapkan berbagai AI transpiling passes tergantung pada `optimization_level` yang diminta. Jika `"false"`, layanan menggunakan fitur transpilasi Qiskit terbaru tanpa AI. Terakhir, jika `"auto"`, layanan memutuskan apakah akan menerapkan passes heuristik Qiskit standar atau AI-powered passes berdasarkan Circuit kamu.
- `qiskit_transpile_options` (dict) - Objek dictionary Python yang bisa menyertakan opsi lain yang valid dalam [metode `transpile()` Qiskit](defaults-and-configuration-options). Jika `qiskit_transpile_options` menyertakan `optimization_level`, nilainya akan diabaikan demi `optimization_level` yang ditentukan sebagai parameter input. Jika `qiskit_transpile_options` menyertakan opsi yang tidak dikenali oleh metode `transpile()` Qiskit, library akan memunculkan error.

Untuk informasi lebih lanjut tentang metode `qiskit-ibm-transpiler` yang tersedia, lihat [referensi API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Untuk mempelajari lebih lanjut tentang API layanan, lihat [dokumentasi REST API Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Contoh
Contoh-contoh berikut menunjukkan cara mentranspilasi Circuit menggunakan Qiskit Transpiler Service dengan berbagai parameter.

1. Buat sebuah Circuit dan panggil Qiskit Transpiler Service untuk mentranspilasi Circuit tersebut dengan `ibm_torino` sebagai `backend_name`, 3 sebagai `optimization_level`, dan tanpa menggunakan AI selama transpilasi.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Catatan*: kamu hanya bisa menggunakan perangkat backend_name yang bisa kamu akses dengan akun IBM Quantum kamu. Selain `backend_name`, `TranspilerService` juga memperbolehkan `coupling_map` sebagai parameter.

2. Buat Circuit serupa dan transpilasikan, dengan meminta kemampuan transpilasi AI dengan mengatur flag `ai` ke `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Buat Circuit serupa dan transpilasikan sambil membiarkan layanan memutuskan apakah akan menggunakan AI-powered transpiling passes.